In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, BatchNormalization, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import BinaryAccuracy, Precision, Recall, AUC
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, precision_recall_curve
import joblib, gc

FEATURES_SELECCIONADAS = [
    'iat', 'rst_count', 'urg_count', 'number', 'variance', 'tot_size',
    'max', 'header_length', 'flow_duration', 'weight', 'rate', 'duration',
    'protocol_type', 'syn_flag_number', 'fin_count', 'syn_count',
    'rst_flag_number', 'ack_count'
]
NOMBRE_CLASE_BENIGNA = 'BenignTraffic'


print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))


In [ ]:
df = pd.read_feather('df_binary_balanced.feather')
df['label_binario'] = (df['label'] != NOMBRE_CLASE_BENIGNA).astype(int)
print('Shape:', df.shape)
print(df['label_binario'].value_counts())


In [ ]:
X = df[FEATURES_SELECCIONADAS].values
y = df['label_binario'].values

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.10, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=2/9, random_state=42, stratify=y_temp)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

del df; gc.collect()

print(f'Train: {len(X_train):,}  Val: {len(X_val):,}  Test: {len(X_test):,}')
print(f'Features: {X_train.shape[1]}')


#### Poda de Magnitud (Weight Pruning)
Se utiliza TensorFlow Model Optimization Toolkit para podar gradualmente los pesos menos importantes durante un reentrenamiento corto

In [ ]:
def focal_loss(gamma=2.0, alpha=0.75):
    """
    Focal Loss binaria.
    gamma > 0 enfoca en ejemplos difíciles.
    alpha = peso clase positiva (malicioso).
    NO combinar con class_weight.
    """
    def _loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        bce = -y_true * tf.math.log(y_pred) - (1.0 - y_true) * tf.math.log(1.0 - y_pred)
        p_t = y_true * y_pred + (1.0 - y_true) * (1.0 - y_pred)
        alpha_t = y_true * alpha + (1.0 - y_true) * (1.0 - alpha)
        return tf.reduce_mean(alpha_t * tf.pow(1.0 - p_t, gamma) * bce)
    return _loss

In [ ]:
import tensorflow as tf
import tensorflow_model_optimization as tfmot

# 1. Cargar tu modelo pre-entrenado (el de 21k parámetros)
base_model = tf.keras.models.load_model('mlp_binario_v2_exp2_focalloss.h5', custom_objects={'_loss': focal_loss(2,0.75)})

# 2. Definir los parámetros de poda (Apuntamos a un 60% de escasez final)
pruning_params = {
      'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(
          initial_sparsity=0.0, 
          final_sparsity=0.60, 
          begin_step=0, 
          end_step=int(163155)) # Ajustar end_step según (muestras / batch_size) * epocas 1383725/128 * 15
}

# 3. Aplicar el wrapper de poda al modelo
pruned_model = tfmot.sparsity.keras.prune_low_magnitude(base_model, **pruning_params)

# 4. Compilar (Con parche para el tipo de dato del optimizador)

# 4.1 Instanciar el optimizador explícitamente
optimizer = tf.keras.optimizers.Adam()

# 4.2 Compilar el modelo podado (sin necesidad de run_eagerly)
pruned_model.compile(optimizer=optimizer, 
                     loss=focal_loss(2,0.75), 
                     metrics=[tf.keras.metrics.AUC()])

# 5. Entrenar (Fine-tuning para acostumbrar la red a la pérdida de conexiones)
callbacks = [tfmot.sparsity.keras.UpdatePruningStep()]
pruned_model.fit(X_train, y_train, epochs=15, callbacks=callbacks, validation_data=(X_val, y_val))

# 6. IMPORTANTE: Remover los wrappers de poda antes de pasar a QAT
stripped_pruned_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

#### Quantization-Aware Training (QAT) sobre el modelo podado
Tomamos el modelo podado y limitamos sus pesos y activaciones usando qkeras

In [ ]:
from qkeras.utils import model_quantize
from qkeras import quantized_bits

# 1. Definir el diccionario de cuantización (Ejemplo: 8 bits para pesos, 4 bits para activaciones)
config_dict = {
    "QDense": {
        "kernel_quantizer": "quantized_bits(8,0,alpha=auto)",
        "bias_quantizer": "quantized_bits(8,0,alpha=auto)"
    },
    "Activation": {
        "default": "quantized_relu(4,2)" # Ajustar según tu función de activación
    }
}

# 2. Convertir el modelo podado a un modelo QKeras
qat_model = model_quantize(stripped_pruned_model, config_dict, 8, default_bias_quantizer=None)

# 3. Reentrenar con QAT
qat_model.compile(optimizer='adam', loss=focal_loss, metrics=[tf.keras.metrics.AUC()])
qat_model.fit(X_train, y_train, epochs=5, validation_data=(X_val, y_val))

# 4. Guardar modelo final para hls4ml
qat_model.save('modelo_pruned_qat.h5')